In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]", "executable": "/home/miakhlov/course-production-verified-20261118/scripts/environment/.venv-chem/bin/python", "platform": "Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.39", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0", "httpcore": "1.0.9", "httpx": "0.28.1", "idna": 

# Разбор A. Кинетика и план эксперимента

Временные данные способны оценить k только при подходящем плане. Гипотеза: конечный выход и пара ранних точек плохо различают скорость и конечное плато. На A изучаем условную кинетику, не приписывая её опубликованной реакции ксантинов. B проектирует измерение превращений аторвастатина.

Сгенерируйте детерминированные точки при k=.02 с небольшим учебным шумом и seed=42. Оцените k через curve_fit, сравните с истинным параметром, построите остатки. Создайте обратимый контроль с плато .25 и покажите систематическое расхождение необратимой модели. Обсудите случайные и систематические источники ошибки.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M09-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
import numpy as np
from scipy.optimize import curve_fit
rng=np.random.default_rng(42);t=np.array([0,10,20,40,60,90,120.])
truth=np.exp(-.02*t);observed=truth+rng.normal(0,.008,len(t))
model=lambda t,k:np.exp(-k*t)
parameters,cov=curve_fit(model,t,observed,p0=[.01],bounds=(0,np.inf))
khat=float(parameters[0]);print({'k_fit_min_inverse':khat,'half_life_min':math.log(2)/khat})


{'k_fit_min_inverse': 0.02022524752951621, 'half_life_min': 34.27138182355415}


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
plateau=.25+.75*np.exp(-.02*t)
wrong,_=curve_fit(model,t,plateau,p0=[.01],bounds=(0,np.inf))
fig,axes=plt.subplots(1,2,figsize=(9,3));axes[0].scatter(t,observed);axes[0].plot(t,model(t,khat))
axes[0].set(xlabel='Time / min',ylabel='Simulated A/A0')
axes[1].axhline(0,color='gray');axes[1].plot(t,plateau-model(t,*wrong),'o-')
axes[1].set(xlabel='Time / min',ylabel='Plateau-model residual');publish_plot(fig,'fit.png')


<Figure size 900x300 with 2 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'simulation','k_fit':khat,'parameter_covariance':cov.tolist(),'seed':42,'experiment':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L02','A',result,ROOT,inputs=[])
print('Versioned evidence:',receipt)


Versioned evidence: /home/miakhlov/course-production-verified-20261118/practicuum/evidence/20261118-full-course-linux/workspace/my_research/evidence/M09-L02/20261118T142010711402Z-A-c761abe2df76405089b0a329e013997f.json


Возврат исходного k проверяет синтетический эксперимент, не стабильность лекарства. Новые данные должны различать конкурирующие модели. В M10 потребуется разделить соединения, чтобы вообще измерять временные концентрации.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?